In [ ]:
import os
import re
import math
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ============================================================
# CLOCKWORK EVALUATION WITHOUT TRIGGER ON AN IMAGE SEQUENCE
# - Processes the ENTIRE sequence with real HOLD/FIRE states
# - Evaluates ONLY frames with annotations (e.g., every 1 s)
# - FIRE every K frames
# - HOLD completely freezes context_path
# - Exports an Excel file with per-frame metrics and the confusion matrix
# ============================================================

# ======================= EDIT ================================
os.chdir('/mmsegmentation')

CONFIG = '/mmsegmentation/zmax_configs/for_test_hasta_26_3/multitask_test_sin_trigger_clean.py'
CKPT   = '/mmsegmentation/work_dirs/multitask_val_fix_wloss/best_cls_acc_cls_top1_iter_25600.pth'
DEVICE = 'cuda:0'

IMG_DIR      = '/0_secuencia_paravideo/1a_secuencia2_paravideo/'
MASK_DIR     = '/0_secuencias_ann/1a_secuencia/ann/'
CLS_ANN_PATH = '/0_secuencias_ann/anotacion_1a_cls_secuencia.txt'
OUTPUT_XLSX  = '/mmsegmentation/output/eval_clockwork_1a_k30.xlsx'

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.bmp'}
MASK_EXT = '.png'
NUM_SEG_CLASSES = 2
IGNORE_INDEX = 255
GT_MASK_BINARY = True

# MODEL class names (classifier order)
MODEL_CLS_NAMES = ['izquierda', 'recta', 'derecha']

# Mapping from GT labels in the txt file to model class names
GT_LABEL_MAP = {
    'LEFT': 'izquierda',
    'STRAIGHT': 'recta',
    'RIGHT': 'derecha',
    'IZQUIERDA': 'izquierda',
    'RECTA': 'recta',
    'DERECHA': 'derecha',
}

# Fixed Clockwork without trigger
KEYFRAME_K = 30
START_FIRE_AT = 0

# Temporal matching between annotations <-> images (seconds)
TIMESTAMP_MATCH_TOL = 1e-4
EVALUATE_ONLY_ANNOTATED_TIMESTAMPS = True

# Inference consistent with the video script
PREPROCESS_USE_PINNED = True
PRED_MASK_DTYPE = np.uint8
USE_AUTOCAST = True
# ============================================================

import os.path as osp
from mmseg.apis import init_model
from mmseg.utils import register_all_modules

try:
    from mmseg.structures.dual_task_seg_data_sample import DualTaskSegDataSample as _TestDataSample
except Exception:
    from mmseg.structures import SegDataSample as _TestDataSample


def ensure_dir_for_file(path: str):
    Path(path).parent.mkdir(parents=True, exist_ok=True)


def parse_timestamp_from_name(path: str) -> Optional[float]:
    stem = Path(path).stem
    try:
        return float(stem)
    except Exception:
        return None


def sorted_image_paths(img_dir: str) -> List[str]:
    paths = [str(p) for p in Path(img_dir).iterdir() if p.suffix.lower() in IMG_EXTS]

    def _key(p: str):
        t = parse_timestamp_from_name(p)
        return (0, t) if t is not None else (1, Path(p).name)

    return sorted(paths, key=_key)


def normalize_name_key(name: str) -> Tuple[str, str]:
    base = os.path.basename(name)
    stem = os.path.splitext(base)[0]
    return base, stem


def normalize_gt_label(raw_label: str) -> str:
    key = str(raw_label).strip().upper()
    if key in GT_LABEL_MAP:
        return GT_LABEL_MAP[key]
    raise KeyError(f'La etiqueta GT "{raw_label}" no existe en GT_LABEL_MAP.')


def ts_key(ts: float) -> int:
    return int(round(float(ts) / TIMESTAMP_MATCH_TOL))


def parse_cls_annotations(path: str):
    ann_by_name = {}
    ann_by_ts = {}
    with open(path, 'r', encoding='utf-8') as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith('#'):
                continue
            parts = re.split(r'[\s,;]+', line)
            if len(parts) < 2:
                continue

            name_tok = parts[0]
            label_tok = parts[1]
            base, stem = normalize_name_key(name_tok)
            ts = parse_timestamp_from_name(name_tok)
            model_label = normalize_gt_label(label_tok)
            model_idx = MODEL_CLS_NAMES.index(model_label)

            info = {
                'raw_label': label_tok,
                'gt_cls_name': model_label,
                'gt_cls_idx': int(model_idx),
                'image_name': base,
                'timestamp_s': ts,
            }
            ann_by_name[base] = info
            ann_by_name[stem] = info
            if ts is not None:
                ann_by_ts[ts_key(ts)] = info
    return ann_by_name, ann_by_ts


def load_gt_mask(mask_path: str) -> np.ndarray:
    gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if gt is None:
        raise FileNotFoundError(f'No se pudo leer mascara: {mask_path}')
    if GT_MASK_BINARY:
        ignore = (gt == IGNORE_INDEX)
        gt = (gt > 0).astype(np.uint8)
        gt[ignore] = IGNORE_INDEX
    return gt


def fast_hist(true_mask: np.ndarray, pred_mask: np.ndarray, n: int, ignore_index: int = 255) -> np.ndarray:
    valid = (true_mask != ignore_index) & (true_mask >= 0) & (true_mask < n)
    hist = np.bincount(
        n * true_mask[valid].astype(int) + pred_mask[valid].astype(int),
        minlength=n ** 2
    ).reshape(n, n)
    return hist


def ious_from_hist(hist: np.ndarray) -> np.ndarray:
    denom = hist.sum(1) + hist.sum(0) - np.diag(hist)
    return np.divide(np.diag(hist), denom, out=np.zeros_like(np.diag(hist), dtype=np.float64), where=denom > 0)


def safe_nanmean(values: pd.Series) -> float:
    vals = pd.to_numeric(values, errors='coerce').to_numpy(dtype=np.float64)
    return float(np.nanmean(vals)) if np.isfinite(vals).any() else math.nan


def ensure_cm_size(cm: np.ndarray, m: int) -> np.ndarray:
    if cm.shape[0] >= m and cm.shape[1] >= m:
        return cm
    out = np.zeros((m, m), dtype=np.int64)
    out[:cm.shape[0], :cm.shape[1]] = cm
    return out


def add_cls_pair(cm: np.ndarray, y_true: int, y_pred: int) -> np.ndarray:
    m = max(int(y_true), int(y_pred)) + 1
    cm = ensure_cm_size(cm, m)
    cm[int(y_true), int(y_pred)] += 1
    return cm


def cls_top1_from_cm(cm: np.ndarray) -> float:
    total = int(cm.sum())
    return float(np.trace(cm) / total) if total > 0 else math.nan


class ContextPathClock:
    """
    FIRE:
      - runs the full context_path and caches its output
    HOLD:
      - directly returns the cache from the last FIRE frame
      - does not recompute any part of context_path
    """
    def __init__(self, bise: torch.nn.Module):
        assert hasattr(bise, 'context_path') and hasattr(bise, 'spatial_path'), \
            'El backbone no parece ser BiSeNetV1 con context_path/spatial_path.'
        self.context_path = bise.context_path
        self.cache = None
        self.hold = False
        self._orig_forward = self.context_path.forward
        self._install()

    def _cache_out(self, out):
        if isinstance(out, (list, tuple)):
            self.cache = tuple(o.detach() if torch.is_tensor(o) else o for o in out)
        elif torch.is_tensor(out):
            self.cache = (out.detach(),)
        else:
            self.cache = out

    def _install(self):
        @torch.inference_mode()
        def wrapped_forward(x):
            if self.hold and (self.cache is not None):
                return self.cache
            out = self._orig_forward(x)
            self._cache_out(out)
            return out
        self.context_path.forward = wrapped_forward

    def set_hold(self, flag: bool):
        self.hold = bool(flag)

    def invalidate(self):
        self.cache = None


class FixedClockScheduler:
    def __init__(self, k=100, start_fire_at=0):
        self.k = max(1, int(k))
        self.next_fire_idx = int(start_fire_at)

    def should_fire(self, frame_idx: int) -> bool:
        return int(frame_idx) >= int(self.next_fire_idx)

    def update_after_inference(self, frame_idx: int, did_fire: bool):
        if did_fire:
            self.next_fire_idx = int(frame_idx) + int(self.k)

    @property
    def active_k(self):
        return self.k


def _label_from_LabelData(lbl):
    import numpy as _np
    if lbl is None:
        return None

    x = lbl
    for _ in range(8):
        if x is None:
            return None
        if torch.is_tensor(x):
            return int(x.reshape(-1)[0].detach().cpu().item())
        if isinstance(x, _np.ndarray):
            return int(x.reshape(-1)[0])
        if isinstance(x, (int, float, bool, _np.integer, _np.floating)):
            return int(x)
        if isinstance(x, (list, tuple)):
            if len(x) == 0:
                return None
            x = x[0]
            continue
        if hasattr(x, 'item') and callable(getattr(x, 'item')):
            try:
                return int(x.item())
            except Exception:
                pass
        next_x = None
        for k in ('label', 'pred_label', 'data', 'value'):
            if hasattr(x, k):
                next_x = getattr(x, k)
                break
        if next_x is x:
            break
        x = next_x
    return None


class NDArrayInferencer:
    def __init__(self, model):
        self.model = model
        self.model.eval()
        self.device = next(self.model.parameters()).device

        cfg_dp = model.cfg.model.get('data_preprocessor', {})
        size = cfg_dp.get('size', (512, 512))
        self.input_w = int(size[0])
        self.input_h = int(size[1])
        self.bgr_to_rgb = bool(cfg_dp.get('bgr_to_rgb', True))
        mean = np.array(cfg_dp.get('mean', [123.675, 116.28, 103.53]), dtype=np.float32)
        std = np.array(cfg_dp.get('std', [58.395, 57.12, 57.375]), dtype=np.float32)

        self.mean = torch.tensor(mean, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)
        self.std = torch.tensor(std, device=self.device, dtype=torch.float32).view(1, 3, 1, 1)

        self._meta = dict(
            ori_shape=(self.input_h, self.input_w),
            img_shape=(self.input_h, self.input_w),
            pad_shape=(self.input_h, self.input_w),
            batch_input_shape=(self.input_h, self.input_w),
            scale_factor=(1.0, 1.0),
            padding_size=[0, 0, 0, 0],
            flip=False,
            flip_direction=None,
        )

        self.use_cuda = (self.device.type == 'cuda')
        self.use_pinned = bool(PREPROCESS_USE_PINNED and self.use_cuda)

        self._resize_hwc = np.empty((self.input_h, self.input_w, 3), dtype=np.uint8)
        self._cpu_hwc_t = None
        self._cpu_hwc_np = None
        self._gpu_hwc_u8 = None
        self._gpu_chw_f32 = None

        if self.use_pinned:
            self._cpu_hwc_t = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, pin_memory=True)
            self._cpu_hwc_np = self._cpu_hwc_t[0].numpy()
            self._gpu_hwc_u8 = torch.empty((1, self.input_h, self.input_w, 3), dtype=torch.uint8, device=self.device)
            self._gpu_chw_f32 = torch.empty((1, 3, self.input_h, self.input_w), dtype=torch.float32, device=self.device)

    def _make_data_sample(self):
        ds = _TestDataSample()
        ds.set_metainfo(self._meta.copy())
        return ds

    def _preprocess(self, img_bgr_nd):
        if (img_bgr_nd.shape[1], img_bgr_nd.shape[0]) != (self.input_w, self.input_h):
            cv2.resize(img_bgr_nd, (self.input_w, self.input_h), dst=self._resize_hwc, interpolation=cv2.INTER_LINEAR)
            img = self._resize_hwc
        else:
            img = img_bgr_nd

        if self.use_pinned:
            np.copyto(self._cpu_hwc_np, img)
            self._gpu_hwc_u8.copy_(self._cpu_hwc_t, non_blocking=True)
            chw_u8 = self._gpu_hwc_u8.permute(0, 3, 1, 2)
            if self.bgr_to_rgb:
                chw_u8 = chw_u8[:, [2, 1, 0], :, :]
            self._gpu_chw_f32.copy_(chw_u8)
            self._gpu_chw_f32.sub_(self.mean).div_(self.std)
            return self._gpu_chw_f32

        if self.bgr_to_rgb:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        tensor = torch.from_numpy(img).unsqueeze(0)
        if self.use_cuda:
            tensor = tensor.to(self.device, non_blocking=True)
        tensor = tensor.float()
        tensor = (tensor - self.mean) / self.std
        return tensor

    @torch.inference_mode()
    def __call__(self, img_bgr_nd):
        inputs = self._preprocess(img_bgr_nd)
        data_samples = [self._make_data_sample()]

        if USE_AUTOCAST and self.use_cuda:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                preds = self.model.predict(inputs, data_samples)
        else:
            preds = self.model.predict(inputs, data_samples)

        sample = preds[0]
        pred = sample.pred_sem_seg.data
        if pred.ndim == 3 and pred.shape[0] == 1:
            pred = pred[0]
        pred = pred.to(dtype=getattr(torch, str(np.dtype(PRED_MASK_DTYPE).name)))
        pred_mask = pred.detach().cpu().numpy()

        cls_idx = None
        if hasattr(sample, 'pred_label') and sample.pred_label is not None:
            cls_idx = _label_from_LabelData(sample.pred_label)
        if cls_idx is None:
            raise RuntimeError('No se encontro pred_label en el DataSample de salida.')

        pred_cls_name = MODEL_CLS_NAMES[int(cls_idx)] if 0 <= int(cls_idx) < len(MODEL_CLS_NAMES) else str(cls_idx)
        return pred_mask, int(cls_idx), pred_cls_name


def find_mask_path(mask_dir: str, img_name: str) -> str:
    cand1 = os.path.join(mask_dir, img_name)
    if os.path.exists(cand1):
        return cand1
    stem = Path(img_name).stem
    cand2 = os.path.join(mask_dir, stem + MASK_EXT)
    if os.path.exists(cand2):
        return cand2
    raise FileNotFoundError(f'No se encontro mascara para {img_name}')


def prepare_model():
    register_all_modules(init_default_scope=False)
    model = init_model(CONFIG, CKPT, device=DEVICE)
    model.eval()

    if hasattr(model, 'backbone') and hasattr(model.backbone, 'context_path'):
        ctx_clock = ContextPathClock(model.backbone)
    else:
        raise RuntimeError('El modelo no parece ser BiSeNetV1 con context_path.')

    infer = NDArrayInferencer(model)
    scheduler = FixedClockScheduler(k=KEYFRAME_K, start_fire_at=START_FIRE_AT)
    return model, ctx_clock, infer, scheduler


def evaluate_sequence():
    img_paths = sorted_image_paths(IMG_DIR)
    if len(img_paths) == 0:
        raise RuntimeError(f'No se encontraron imagenes en {IMG_DIR}')

    ann_by_name, ann_by_ts = parse_cls_annotations(CLS_ANN_PATH)
    _, ctx_clock, infer, scheduler = prepare_model()

    results = []
    cls_cm = np.zeros((len(MODEL_CLS_NAMES), len(MODEL_CLS_NAMES)), dtype=np.int64)
    seg_hist_total = np.zeros((NUM_SEG_CLASSES, NUM_SEG_CLASSES), dtype=np.int64)

    t0 = parse_timestamp_from_name(img_paths[0])

    for idx, img_path in enumerate(tqdm(img_paths, desc='Evaluando secuencia clockwork sin gatillo')):
        img_name = os.path.basename(img_path)
        timestamp = parse_timestamp_from_name(img_path)
        time_rel_s = (timestamp - t0) if (timestamp is not None and t0 is not None) else float(idx)

        do_fire = scheduler.should_fire(idx)
        ctx_clock.set_hold(not do_fire)
        state = 'FIRE' if do_fire else 'HOLD'

        img_bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'No se pudo leer imagen: {img_path}')

        pred_mask_small, pred_cls_idx, pred_cls_name = infer(img_bgr)
        scheduler.update_after_inference(idx, did_fire=do_fire)

        # Match GT only if the timestamp is annotated
        gt_info = None
        if timestamp is not None:
            gt_info = ann_by_ts.get(ts_key(timestamp), None)
        evaluated = bool(gt_info is not None) if EVALUATE_ONLY_ANNOTATED_TIMESTAMPS else True

        row = {
            'frame_idx': int(idx),
            'image_name': img_name,
            'timestamp_s': float(timestamp) if timestamp is not None else np.nan,
            'time_rel_s': float(time_rel_s),
            'mode': state,
            'is_fire': int(do_fire),
            'is_hold': int(not do_fire),
            'evaluated_with_gt': int(evaluated),
            'pred_cls_idx': int(pred_cls_idx),
            'pred_cls_name': pred_cls_name,
            'gt_cls_idx': np.nan,
            'gt_cls_name': None,
            'cls_correct': np.nan,
            'seg_mIoU': np.nan,
            'seg_IoU_background': np.nan,
            'seg_IoU_road': np.nan,
            'mask_path': None,
            'gt_label_raw': None,
        }

        if evaluated:
            if gt_info is None:
                raise RuntimeError(f'El frame {img_name} quedo marcado para evaluar, pero no se encontro GT de clasificacion.')

            mask_path = find_mask_path(MASK_DIR, gt_info['image_name'])
            gt_mask = load_gt_mask(mask_path)
            pred_mask = cv2.resize(pred_mask_small.astype(np.uint8), (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)

            hist = fast_hist(gt_mask, pred_mask, NUM_SEG_CLASSES, IGNORE_INDEX)
            seg_hist_total += hist
            ious = ious_from_hist(hist)
            miou = float(np.nanmean(ious))
            iou_bg = float(ious[0]) if len(ious) > 0 else math.nan
            iou_road = float(ious[1]) if len(ious) > 1 else math.nan

            gt_cls_idx = int(gt_info['gt_cls_idx'])
            gt_cls_name = str(gt_info['gt_cls_name'])
            cls_cm = add_cls_pair(cls_cm, gt_cls_idx, pred_cls_idx)

            row.update({
                'gt_cls_idx': gt_cls_idx,
                'gt_cls_name': gt_cls_name,
                'cls_correct': int(gt_cls_idx == int(pred_cls_idx)),
                'seg_mIoU': miou,
                'seg_IoU_background': iou_bg,
                'seg_IoU_road': iou_road,
                'mask_path': mask_path,
                'gt_label_raw': gt_info['raw_label'],
            })

        results.append(row)

    df = pd.DataFrame(results)
    eval_df = df[df['evaluated_with_gt'] == 1].copy()
    seg_ious_total = ious_from_hist(seg_hist_total)

    summary = {
        'n_all_frames': int(len(df)),
        'n_evaluated_frames': int(len(eval_df)),
        'fire_count_all': int((df['mode'] == 'FIRE').sum()),
        'hold_count_all': int((df['mode'] == 'HOLD').sum()),
        'fire_count_evaluated': int(((eval_df['mode'] == 'FIRE')).sum()),
        'hold_count_evaluated': int(((eval_df['mode'] == 'HOLD')).sum()),
        'mean_frame_mIoU_eval_only': safe_nanmean(eval_df['seg_mIoU']) if len(eval_df) else math.nan,
        'mean_frame_IoU_background_eval_only': safe_nanmean(eval_df['seg_IoU_background']) if len(eval_df) else math.nan,
        'mean_frame_IoU_road_eval_only': safe_nanmean(eval_df['seg_IoU_road']) if len(eval_df) else math.nan,
        'global_mIoU_from_total_hist_eval_only': float(np.nanmean(seg_ious_total)) if len(eval_df) else math.nan,
        'global_IoU_background_eval_only': float(seg_ious_total[0]) if len(seg_ious_total) > 0 and len(eval_df) else math.nan,
        'global_IoU_road_eval_only': float(seg_ious_total[1]) if len(seg_ious_total) > 1 and len(eval_df) else math.nan,
        'cls_top1_eval_only': float(cls_top1_from_cm(cls_cm)),
        'keyframe_k': int(KEYFRAME_K),
        'timestamp_match_tol_s': float(TIMESTAMP_MATCH_TOL),
        'config': CONFIG,
        'checkpoint': CKPT,
        'img_dir': IMG_DIR,
        'mask_dir': MASK_DIR,
        'cls_ann_path': CLS_ANN_PATH,
    }
    return df, eval_df, cls_cm, summary


def style_sheet_basic(ws):
    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    center = Alignment(horizontal='center', vertical='center')
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = center
    ws.freeze_panes = 'A2'
    for col_cells in ws.columns:
        length = 0
        col_letter = get_column_letter(col_cells[0].column)
        for c in col_cells:
            try:
                length = max(length, len(str(c.value)) if c.value is not None else 0)
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = min(max(length + 2, 10), 42)


def write_excel(df: pd.DataFrame, eval_df: pd.DataFrame, cls_cm: np.ndarray, summary: Dict[str, object], out_path: str):
    ensure_dir_for_file(out_path)

    cm_df = pd.DataFrame(
        cls_cm,
        index=[f'GT_{c}' for c in MODEL_CLS_NAMES],
        columns=[f'Pred_{c}' for c in MODEL_CLS_NAMES],
    )
    summary_df = pd.DataFrame({'metric': list(summary.keys()), 'value': list(summary.values())})

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name='PerFrame', index=False)
        eval_df.to_excel(writer, sheet_name='EvaluatedOnly', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)
        cm_df.to_excel(writer, sheet_name='ConfusionMatrix')

    wb = load_workbook(out_path)
    for sheet_name in ['PerFrame', 'EvaluatedOnly', 'Summary', 'ConfusionMatrix']:
        ws = wb[sheet_name]
        style_sheet_basic(ws)

    for sheet_name in ['PerFrame', 'EvaluatedOnly']:
        ws = wb[sheet_name]
        num_cols = {}
        headers = {cell.value: cell.column for cell in ws[1] if cell.value}
        for col_name, fmt in {
            'timestamp_s': '0.000000',
            'time_rel_s': '0.000',
            'seg_mIoU': '0.0000',
            'seg_IoU_background': '0.0000',
            'seg_IoU_road': '0.0000',
        }.items():
            if col_name in headers:
                col_letter = get_column_letter(headers[col_name])
                for cell in ws[col_letter][1:]:
                    cell.number_format = fmt

    ws = wb['PerFrame']
    start_row = ws.max_row + 3
    ws.cell(start_row, 1, 'Matriz de confusion de clasificacion (solo frames evaluados)')
    ws.cell(start_row, 1).font = Font(bold=True)
    for j, name in enumerate([''] + [f'Pred_{c}' for c in MODEL_CLS_NAMES], start=1):
        ws.cell(start_row + 1, j, name)
        ws.cell(start_row + 1, j).font = Font(bold=True)
    for i, gt_name in enumerate(MODEL_CLS_NAMES, start=0):
        ws.cell(start_row + 2 + i, 1, f'GT_{gt_name}')
        ws.cell(start_row + 2 + i, 1).font = Font(bold=True)
        for j, _ in enumerate(MODEL_CLS_NAMES, start=0):
            ws.cell(start_row + 2 + i, 2 + j, int(cls_cm[i, j]))

    sr = start_row + 2 + len(MODEL_CLS_NAMES) + 3
    ws.cell(sr, 1, 'Resumen')
    ws.cell(sr, 1).font = Font(bold=True)
    for k, (metric, value) in enumerate(summary.items(), start=1):
        ws.cell(sr + k, 1, metric)
        ws.cell(sr + k, 2, value)

    wb.save(out_path)


def main():
    df, eval_df, cls_cm, summary = evaluate_sequence()
    write_excel(df, eval_df, cls_cm, summary, OUTPUT_XLSX)

    print('\n=== RESUMEN ===')
    for k, v in summary.items():
        print(f'{k}: {v}')
    print(f'\nExcel guardado en: {OUTPUT_XLSX}')
    return df, eval_df, cls_cm, summary


# Run
results_df, eval_df, cls_cm, summary = main()
